In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
%pip install yfinance requests --quiet

StatementMeta(, 22c35f5b-1041-4356-97d6-7d01f1ea833f, 7, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [12]:
import datetime

# ── Telegram ──────────────────────────────────────────────────────────────────
TELEGRAM_TOKEN   = "8659001033:AAG9wyr_CdGgL0hvY6C1LQ53jrr5IRFvqFA"          # your bot token
TELEGRAM_CHAT_ID = "-5200409875"                        # your group chat id

# ── Market tickers ────────────────────────────────────────────────────────────
TICKERS = {
    "nasdaq_price_usd":            ("^IXIC",     "NASDAQ",    "🇺🇸", "USD"),
    "sp500_price_usd":             ("^GSPC",     "S&P 500",   "🇺🇸", "USD"),
    "dowjones_price_usd":          ("^DJI",      "Dow Jones", "🇺🇸", "USD"),
    "china_shanghai_price_usd":    ("000001.SS", "Shanghai",  "🇨🇳", "USD"),
    "hongkong_hongkong_price_usd": ("^HSI",      "HK",        "🇭🇰", "USD"),
    "uk_london_price_usd":         ("^FTSE",     "London",    "🇬🇧", "USD"),
    "japan_tokyo_price_usd":       ("^N225",     "Tokyo",     "🇯🇵", "USD"),
    "egx30_price_egp":             ("^CASE30",   "EGX30",     "🇪🇬", "EGP"),
    "brent_oil_price_usd":         ("BZ=F",      "Brent Oil", "🛢️",  "USD"),
    "gold_price_usd":              ("GC=F",      "Gold",      "🥇",  "USD"),
}

StatementMeta(, 22c35f5b-1041-4356-97d6-7d01f1ea833f, 19, Finished, Available, Finished, False)

In [13]:
import yfinance as yf
import json
from datetime import datetime, timezone

def fetch_prices():
    prices = {}
    now    = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    for col, (ticker, label, flag, ccy) in TICKERS.items():
        try:
            price = yf.Ticker(ticker).fast_info.last_price
            prices[col] = {"label": label, "flag": flag,
                           "currency": ccy, "price": round(float(price), 2)}
        except Exception as e:
            prices[col] = {"label": label, "flag": flag,
                           "currency": ccy, "price": None}
            print(f"  [WARN] {label}: {e}")
    return prices, now

prices, fetch_time = fetch_prices()
print(f"Fetched at {fetch_time}")
for col, v in prices.items():
    print(f"  {v['flag']} {v['label']:<14} {v['price']:>12,.2f} {v['currency']}" if v['price'] else f"  {v['flag']} {v['label']:<14} N/A")

StatementMeta(, 22c35f5b-1041-4356-97d6-7d01f1ea833f, 20, Finished, Available, Finished, False)

Fetched at 2026-05-10 18:00 UTC
  🇺🇸 NASDAQ            26,247.08 USD
  🇺🇸 S&P 500            7,398.93 USD
  🇺🇸 Dow Jones         49,609.16 USD
  🇨🇳 Shanghai           4,179.95 USD
  🇭🇰 HK                26,393.71 USD
  🇬🇧 London            10,233.10 USD
  🇯🇵 Tokyo             62,713.65 USD
  🇪🇬 EGX30             54,628.70 EGP
  🛢️ Brent Oil            101.29 USD
  🥇 Gold               4,730.70 USD


In [14]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ── build rows ────────────────────────────────────────────────────────────────
rows = []

for col, v in prices.items():
    rows.append({
        "fetch_time": fetch_time,
        "col_name": col,
        "label": v["label"],
        "currency": v["currency"],
        "price": float(v["price"]) if v["price"] is not None else None,
    })

# convert to spark dataframe
df_spark = spark.createDataFrame(pd.DataFrame(rows))

# write to delta table
df_spark.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("HourlyPrices")

print(f"✓ {len(rows)} prices saved to HourlyPrices at {fetch_time}")

StatementMeta(, 22c35f5b-1041-4356-97d6-7d01f1ea833f, 21, Finished, Available, Finished, False)

✓ 10 prices saved to HourlyPrices at 2026-05-10 18:00 UTC


In [15]:
import requests

def send_telegram(token, chat_id, message):
    url = f"https://api.telegram.org/bot{token}/sendMessage"

    resp = requests.post(
        url,
        json={
            "chat_id": chat_id,
            "text": message,
            "parse_mode": "Markdown"
        },
        timeout=10
    )

    return resp.status_code == 200


# ── Build telegram message ────────────────────────────────────────────────────
lines = [
    f"📊 *Market Live Prices*",
    f"🕐 {fetch_time}",
    ""
]

for col, v in prices.items():

    if v["price"] is not None:
        price_str = f"{v['price']:,.2f} {v['currency']}"
    else:
        price_str = "N/A"

    lines.append(
        f"{v['flag']} *{v['label']}*  `{price_str}`"
    )


# ── Append today's predictions if table exists ───────────────────────────────
try:

    preds_df = spark.sql("""
        SELECT
            label,
            currency,
            predicted_tomorrow
        FROM DailyPredictions
        WHERE prediction_date = current_date()
    """).toPandas()

    if len(preds_df) > 0:

        lines.append("")
        lines.append("📈 *Tomorrow's Predictions:*")

        for _, row in preds_df.iterrows():

            pred_str = f"{row['predicted_tomorrow']:,.2f} {row['currency']}"

            lines.append(
                f"→ {row['label']}: `{pred_str}`"
            )

except Exception as e:
    print(f"Predictions skipped: {e}")


# ── Send message ──────────────────────────────────────────────────────────────
message = "\n".join(lines)

tg_ok = send_telegram(
    TELEGRAM_TOKEN,
    TELEGRAM_CHAT_ID,
    message
)

print(f"✓ Telegram: {'sent' if tg_ok else 'FAILED'}")


# ── Exit notebook for pipeline success ───────────────────────────────────────
notebookutils.notebook.exit("success")

StatementMeta(, 22c35f5b-1041-4356-97d6-7d01f1ea833f, 22, Finished, Available, Finished, False)

✓ Telegram: sent
ExitValue: success